<a href="https://colab.research.google.com/github/gmauricio-toledo/math-ML/blob/main/Notebooks/19-Limitaciones_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Limitaciones de las redes MLP</h1>

En esta notebook mostraremos algunos problemas específicos que se benefician de arquitecturas especificas:

* Redes convolucionales (CNN) para imágenes
* Transformers para texto
* Redes recurrentes (RNN) para secuencias temporales

# Imágenes

In [ ]:
!gdown 1gtTqIeOBTuhDlynLDwc3hxDciqT2JBtj
!unzip cifar-10.zip -d /cifar10

In [ ]:
import pandas as pd

labels_df = pd.read_csv('/cifar10/trainLabels.csv',index_col=0)
labels_df

In [ ]:
labels_dict = {j:label for j,label in enumerate(labels_df['label'].unique())}
labels_dict

In [ ]:
import os
import glob
import numpy as np
from PIL import Image

# Buscar todos los archivos .png en el directorio /cifar10 y subdirectorios
ruta_imagenes = glob.glob('/cifar10/**/*.png', recursive=True)

lista_imagenes = []
lista_etiquetas = []

# Crear un diccionario inverso para mapear el string de la etiqueta a un entero
inv_labels_dict = {v: k for k, v in labels_dict.items()}

for ruta in ruta_imagenes:
    # Abrir imagen y asegurar que esté en formato RGB
    img = Image.open(ruta).convert('RGB')
    img_array = np.array(img)

    # Validar que tenga las dimensiones correctas antes de agregarla
    if img_array.shape == (32, 32, 3):
        lista_imagenes.append(img_array)

        # Obtener el ID de la imagen a partir del nombre del archivo (sin la extensión)
        img_id = int(os.path.splitext(os.path.basename(ruta))[0])

        # Obtener la etiqueta en formato string desde labels_df usando el ID
        label_str = labels_df.loc[img_id, 'label']

        # Mapear el string al entero correspondiente y agregarlo a la lista de etiquetas
        lista_etiquetas.append(inv_labels_dict[label_str])

# Convertir las listas en arreglos de NumPy
X_tensor = np.stack(lista_imagenes)
y_tensor = np.array(lista_etiquetas)

print(f"Número total de imágenes procesadas: {len(lista_imagenes)}")
print(f"Dimensiones del tensor 4D X: {X_tensor.shape}")
print(f"Dimensiones del vector de etiquetas y: {y_tensor.shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Seleccionar 6 índices aleatorios del conjunto de entrenamiento
indices_aleatorios = np.random.choice(len(X_train), 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(4, 5))

for i, ax in enumerate(axes.flat):
    idx = indices_aleatorios[i]
    img = X_train[idx]
    label_idx = y_train[idx]

    # Obtener el nombre de la clase desde el diccionario
    class_name = labels_dict[label_idx]

    ax.imshow(img)
    ax.set_title(f'Clase: {class_name}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Crear un mapeo de índices a los nombres reales de las clases
labels_names = [labels_dict[i] for i in y_tensor]

# Contar la frecuencia de cada clase utilizando pandas
counts = pd.Series(labels_names).value_counts()

# Crear el gráfico de barras
plt.figure(figsize=(10, 5))
sns.barplot(x=counts.index, y=counts.values, palette='viridis', hue=counts.index, legend=False)
plt.title('Distribución de Clases en CIFAR-10')
plt.xlabel('Clase')
plt.ylabel('Cantidad de Imágenes')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Mostrar conteo en texto
print(counts)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_tensor,
                                                    y_tensor,
                                                    test_size=0.2,
                                                    stratify=y_tensor,
                                                    random_state=42)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
import matplotlib.pyplot as plt
import numpy as np

# 1. Preprocesamiento
# Normalizar las imágenes a [0, 1] para mejorar el desempeño de la red
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

# Convertir las etiquetas a one-hot encoding para usar categorical_crossentropy y la métrica F1Score
num_classes = 10
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# 2. Definición del modelo MLP (sin convoluciones)
model_mlp = Sequential([
    Flatten(input_shape=(32, 32, 3)),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])

# Usar F1Score de Keras (promedio macro para tener en cuenta todas las clases de manera equitativa)
f1_metric = tf.keras.metrics.F1Score(average='macro', name='f1_score')

model_mlp.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy', f1_metric])

In [ ]:
# 3. Entrenamiento
history = model_mlp.fit(X_train_norm, y_train_cat,
                        epochs=15,
                        batch_size=64,
                        validation_data=(X_test_norm, y_test_cat))

In [ ]:
# 4. Gráficas del entrenamiento
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ax[0].plot(history.history['loss'], label='Train Loss')
ax[0].plot(history.history['val_loss'], label='Val Loss')
ax[0].set_title('Pérdida (Loss)')
ax[0].legend()
ax[0].grid(True)

ax[1].plot(history.history['accuracy'], label='Train Acc')
ax[1].plot(history.history['val_accuracy'], label='Val Acc')
ax[1].set_title('Precisión (Accuracy)')
ax[1].legend()
ax[1].grid(True)

ax[2].plot(history.history['f1_score'], label='Train F1')
ax[2].plot(history.history['val_f1_score'], label='Val F1')
ax[2].set_title('F1 Score')
ax[2].legend()
ax[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score

# 5. Evaluación y Matriz de confusión en el Test
y_pred_prob = model_mlp.predict(X_test_norm)
y_pred = np.argmax(y_pred_prob, axis=1)

print(f"F1 Score en el conjunto de prueba: {f1_score(y_test, y_pred, average='macro')}")
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
# Usamos los nombres de las clases obtenidos anteriormente del diccionario
class_names = [labels_dict[i] for i in range(num_classes)]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Matriz de Confusión - MLP sobre CIFAR-10')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

# NLP

In [ ]:
!gdown 17W3bObZBL6XQVsVMSvkNildhsFSjsmhV

In [ ]:
import pandas as pd

imdb_df = pd.read_csv('IMDB50K.csv')
imdb_df

In [ ]:
y = imdb_df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)
display(y.value_counts())
y = y.values

In [ ]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stopwords = stopwords.words('english')
stopwords[:15]

In [ ]:
from sklearn.model_selection import train_test_split

X_train_df, X_test_df, y_train, y_test = train_test_split(imdb_df[['review']],
                                                    y,
                                                    test_size=0.2,
                                                    stratify=y,
                                                    random_state=42)

In [ ]:
X_train_df

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=2000)
X_train = vectorizer.fit_transform(X_train_df['review'])
X_train = X_train.todense()
X_test = vectorizer.transform(X_test_df['review'])
X_test = X_test.todense()

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential()
model.add(Dense(1024, input_shape=(2000,), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
# model.add(Dense(256, activation='relu'))
# model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
history = model.fit(X_train, y_train,
                    epochs=10,
                    batch_size=32,
                    validation_data=(X_test, y_test))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc='upper right')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuray')
plt.legend(loc='upper right')

plt.show()

In [ ]:
model.evaluate(X_test, y_test)

# Secuencias y Series de Tiempo

In [ ]:
!gdown 10nU_BQ7OHGzbl_T9N7_hnM6KJ6to_4FG

Usaremos un conjunto de datos para predecir el número de pasajeros de líneas aéreas internacionales. Se trata de un problema en el que, dados un año y un mes, la tarea consiste en predecir el número de pasajeros de líneas aéreas internacionales en unidades de 1.000. Los datos van de enero de 1949 a diciembre de 1960, es decir, 12 años, con 144 observaciones. El conjunto original de datos se puede encontrar [aquí](https://www.kaggle.com/datasets/rakannimer/air-passengers).

In [ ]:
import pandas as pd

url = "AirPassengers.txt"

dataframe = pd.read_csv(url)
display(dataframe)

Visualizamos los datos

In [ ]:
# ----- Cambiamos el índice del dataframe por la fecha de la observación -----
dataframe.columns = ['Date', 'Passengers']
dataframe['Date'] = pd.to_datetime(dataframe['Date'], format='%Y-%m')
dataframe = dataframe.set_index('Date')
dataframe.head()

plt.plot(dataframe, color='black')
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.show()

Nos quedamos con la serie de valores del dataframe

In [ ]:
dataset = dataframe.values
dataset = dataset.astype('float32')
dataset

## Preprocesamiento de los datos

Necesitamos generar los conjunto $X$, $Y$ para la red LSTM. El conjunto $X$ es el conjunto que ya tenemos de datos de la serie de tiempo. El conjunto $y$ es la misma serie de tiempo, *adelantada* $l_b=1$ observación. Es decir, si la serie de tiempo esta dada por $x_0,x_1,...,x_n$, tendremos:

$$X = (x_0,x_1,...,x_{n-2})$$
$$Y = (x_1,x_2,...,x_{n-1})$$

Podemos variar el parámetro $l_b$ (look back). **Ahorita no lo haremos, sino hasta el final del ejemplo**

Definimos una función que genere los arreglos $X$, $Y$.

Con esta funcion transformamos la serie de tiempo en un formato supervisado donde $X$ contiene las observaciones pasadas y $Y$ los valores a predecir.

In [ ]:
import numpy as np

def create_dataset(x, look_back=1):
    """
    Transforma una serie de tiempo en un conjunto de datos estructurado para
    aprendizaje supervisado (ideal para entrenar modelos como MLP, LSTM, etc.).

    Toma una secuencia de datos y la divide en "ventanas" (características/X)
    y el valor inmediatamente siguiente a cada ventana (objetivo/Y).

    Parámetros:
    x (array-like): El conjunto de datos de la serie de tiempo (se asume que es
                    un arreglo 2D donde la información está en la columna 0).
    look_back (int): El número de pasos de tiempo previos que se usarán como
                     variables de entrada para predecir el siguiente periodo.

    Retorna:
    tuple: Dos arreglos de NumPy (dataX, dataY) listos para entrenar la red.
    """

    # Inicializamos dos listas vacías: dataX para las entradas (features) y dataY para las salidas esperadas (targets)
    dataX, dataY = [], []

    # Iteramos sobre los datos. El límite es la longitud de 'x' menos 'look_back' menos 1.
    # Esto es crucial para evitar un error de "índice fuera de rango" al intentar acceder al valor a predecir al final de la serie.
    for i in range(len(x) - look_back - 1):

        # Extraemos una "ventana" de tamaño 'look_back' desde el índice 'i'.
        # El ', 0' indica que estamos tomando la primera columna del arreglo (asumiendo forma [muestras, características]).
        a = x[i:(i + look_back), 0]

        # Agregamos esta secuencia de entrada a nuestra lista de características X
        dataX.append(a)

        # El valor objetivo (Y) que la red debe predecir es el valor que está justo después de la ventana 'a'
        dataY.append(x[i + look_back, 0])

    # Finalmente, convertimos las listas de Python a arreglos de NumPy, que es el formato requerido por Keras/TensorFlow/PyTorch
    return np.array(dataX), np.array(dataY)

**Regresaremos a probar con varios valores de *look_back* $l_b$ al final del ejercicio, por ahora, nos quedamos con $l_b=3$.**

In [ ]:
look_back = 3
# look_back = 5

In [ ]:
import numpy as np

print(f"Tamaño de la serie inicial: {dataset.shape[0]}")
print(f"Serie de tiempo original: {dataset[:3].reshape(-1,)} ... {dataset[-4:].reshape(-1,)}")

X, y = create_dataset(dataset,look_back)

print(X.shape, y.shape)

if look_back==1:
    print(f"\nTamaño de las series para la LSTM: {X.shape[0]}")
    print(f"Serie de tiempo modificada X: {X[:3].reshape(-1,)} ... {X[-3:].reshape(-1,)}")
    print(f"Serie de tiempo modificada Y: {y[:3].reshape(-1,)} ... {y[-3:].reshape(-1,)}")
else:
    print(f"\nTamaño de las series para la LSTM: {X.shape}")
    print(f"Serie de tiempo modificada X:\n {X[:3]}")
    print(f"Serie de tiempo modificada Y: {y[:3]}")

Re-escalamos los datos para que queden en el intervalo $[0,1]$.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(dataset)
dataset[:5]

Dividimos en entrenamiento y prueba. Observar que los datos deben ser divididos en forma *ordenada*, es decir:

$$X_{train} = \{ x_0, ... , x_k\} $$
$$X_{test} = \{ x_{k+1}, ... , x_n\} $$

Divide los datos manteniendo el orden temporal, crucial para series de tiempo donde la secuencia importa.

In [ ]:
train_ratio = 0.7

train_size = int(len(dataset) * train_ratio)
test_size = len(dataset) - train_size
train, test = dataset[0:train_size,:], dataset[train_size:len(dataset),:]

# ----- Generamos los conjuntos X,Y de entrenamiento y prueba -----
X_train, Y_train = create_dataset(train, look_back)
X_test, Y_test = create_dataset(test, look_back)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

### Creación y entrenamiento del modelo MLP

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

# Definir el modelo MLP
model_mlp = Sequential()
model_mlp.add(Dense(64, activation='relu', input_dim=look_back))
model_mlp.add(Dense(32, activation='relu'))
model_mlp.add(Dense(1))

model_mlp.compile(optimizer='adam', loss='mean_squared_error')

model.summary()

In [ ]:
history_mlp = model_mlp.fit(X_train, Y_train,
                            epochs=100,
                            batch_size=8,
                            validation_data=(X_test, Y_test),
                            verbose=1)

In [ ]:
# Graficar la pérdida
plt.figure(figsize=(8, 4))
plt.plot(history_mlp.history['loss'], label='Pérdida de Entrenamiento')
plt.plot(history_mlp.history['val_loss'], label='Pérdida de Validación')
plt.title('Curva de Pérdida del Modelo MLP')
plt.xlabel('Épocas')
plt.ylabel('MSE')
plt.legend()
plt.grid(True)
plt.show()

### Predicción Autoregresiva

In [ ]:
# Predicción autoregresiva sobre el conjunto de prueba
predictions_mlp = []
current_window = X_test[0]  # Tomamos la primera ventana del conjunto de prueba como semilla

for i in range(len(X_test)):
    # Predecir el siguiente valor (el modelo espera un batch de 2D)
    pred = model_mlp.predict(current_window.reshape(1, look_back), verbose=0)
    pred_value = pred[0, 0]
    predictions_mlp.append(pred_value)

    # Actualizar la ventana desplazándola e insertando la nueva predicción al final
    current_window = np.append(current_window[1:], pred_value)

predictions_mlp = np.array(predictions_mlp).reshape(-1, 1)

# Revertir la escala para regresar a la magnitud original de pasajeros
predictions_mlp_inv = scaler.inverse_transform(predictions_mlp)
Y_test_inv = scaler.inverse_transform(Y_test.reshape(-1, 1))
Y_train_inv = scaler.inverse_transform(Y_train.reshape(-1, 1))

# Graficar los resultados
plt.figure(figsize=(12, 6))

# Crear los ejes de tiempo
t_train = np.arange(len(Y_train_inv))
t_test = np.arange(len(Y_train_inv), len(Y_train_inv) + len(Y_test_inv))

# Graficar entrenamiento
plt.plot(t_train, Y_train_inv, label='Entrenamiento', color='blue')

# Graficar valores reales de prueba
plt.plot(t_test, Y_test_inv, label='Valores Reales (Prueba)', color='green')

# Graficar predicciones autoregresivas
plt.plot(t_test, predictions_mlp_inv, label='Predicciones MLP (Autoregresivo)', color='red', linestyle='--')

plt.title('Predicción de Pasajeros de Aerolíneas usando MLP (Enfoque Autoregresivo)')
plt.xlabel('Meses (Índice)')
plt.ylabel('Número de Pasajeros')
plt.legend()
plt.grid(True)
plt.show()